# Write a real ADR + a C4 sketch

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 27 §§27.1, 27.5, 27.7, 27.9 · type: worksheet

**The promise:** by the end you have two durable artifacts you keep using — a complete **ADR** for an expensive decision and a **C4** Context/Container sketch of the capstone — and you've self-audited them with the book's review checklist.

This is a **worksheet**: mostly prompts and fill-in markdown cells, with little executable code (one *optional* diagram cell at the end). It runs **fully offline** — there is nothing to call and no key to set.

## 🧠 Why this matters

Architectural judgment — ranking the -ilities, drawing boundaries, recording the *why* — is exactly the skill that **can't be automated or learned by executing cells**. A model will happily generate code at the bottom of the C4 model; what stays human is the top: deciding *what systems exist* and *where the boundaries fall*. So this notebook makes you **decide** and **write**, not run.

The two artifacts here are the ones a team actually lives by. An ADR captures a single expensive decision and its reasoning so it survives turnover and stops you re-litigating settled questions (§27.7). A C4 sketch shows the system at the right altitude for the audience (§27.7). Producing them — and reviewing them against a checklist (§27.9) — is reps for taste, judgment, and review-at-scale: the three skills that *compound* as AI writes more code (§27.8).

## Objectives & prereqs

**By the end you can:**
- Sort mixed decisions into *implementation* (cheap, reversible) vs *architecture* (expensive, one-way door) — the §27.1 cost-to-reverse lens.
- Draw boundaries around **business capabilities**, not technical layers (§27.5).
- Write a full **ADR** in the book's ADR-014 shape (Status / Context / Decision / Consequences) (§27.7).
- Sketch the capstone's **C4** Context and Container levels and run the §27.9 review checklist on your own work.

**Prereqs:** [`27-01-trade-offs-and-ilities.ipynb`](27-01-trade-offs-and-ilities.ipynb) — you'll record the decision you reached there.

**Packages:** Python standard library only. The single optional diagram cell degrades gracefully (prints the diagram source as text) if no renderer is present.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# A worksheet has nothing to call, so MOCK stays the project default (1): fully offline,
# free, deterministic. The switch is kept for consistency with the rest of the course.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

print(f"MOCK mode: {MOCK}  (offline worksheet — no model calls, no key required)")

## 🧠 Architecture = the decisions expensive to change

The most practical definition: **architecture is the set of decisions that are expensive to change later** (§27.1). A variable name is cheap to reverse — *implementation*. A service boundary or a data model is a one-way door — *architecture*. Seniority is largely the skill of telling the two apart and spending your attention accordingly.

Below is a mixed list. Run the cell to see it, then **fill in your sort** in the markdown cell that follows — and, crucially, a one-line *why* for each ("cheap to reverse" or "one-way door").

In [ ]:
decisions = [
    "Rename the `get_user` helper to `fetch_user`",
    "Split billing into its own service with its own database",
    "Choose Postgres vs DynamoDB for the system of record",
    "Pick the variable name for a loop counter",
    "Adopt eventual consistency between orders and inventory",
    "Reorder two function arguments",
    "Expose the agent platform as a public REST API contract v1",
    "Add a debug log line",
    "Make the agent runtime multi-tenant from day one",
    "Switch a JSON field name in an internal-only response",
]
for i, d in enumerate(decisions, 1):
    print(f"{i:2d}. {d}")

### ✍️ Your sort: implementation vs architecture

Move each number into one column and add a one-line *why* (cost to reverse). *(Sample answers shown — replace with your own reasoning.)*

| # | Implementation (cheap, reversible) — *why* | # | Architecture (expensive, one-way door) — *why* |
|---|---|---|---|
| 1 | Rename helper — local edit, IDE refactor | 2 | Split billing out — data + boundary + deploy, very hard to undo |
| 4 | Loop var name — invisible outside the function | 3 | System-of-record DB — migrating data + queries is ruinous |
| 6 | Arg reorder — one call-site sweep | 5 | Eventual consistency — reshapes every consumer's assumptions |
| 8 | Debug log — add/remove freely | 7 | Public API v1 contract — external clients depend on it |
| 10 | Internal JSON field — no external consumer | 9 | Multi-tenant from day one — touches data model, auth, isolation |

> **Rule of thumb:** if undoing it is a pull request, it's implementation; if undoing it is a *migration plus a meeting*, it's architecture — write it down in an ADR.

## Boundaries drill: capabilities, not layers (§27.5)

Put things that change together *together* (**cohesion**) and minimize the dependencies *between* the groups (**coupling**). The hard part is *where the boundaries go* — and the rule is to draw them around **business capabilities** (orders, billing, search), **not** technical layers (controllers, services, repositories), so a typical change lands inside one boundary instead of rippling across all of them.

Here is a feature list for the book's **agent platform** capstone. Run the cell, then fill in the boundaries below.

In [ ]:
features = [
    "Run an agent turn (plan -> tool calls -> answer)",
    "Retrieve documents for grounding (RAG)",
    "Meter token usage and produce an invoice",
    "Register / authenticate a user and an API key",
    "Store and recall long-term conversation memory",
    "Apply a per-tenant rate limit and quota",
    "Index new documents into the vector store",
    "Charge a credit card and record the payment",
    "Stream tokens back to the caller",
    "Record traces/metrics for each agent run",
]
for i, f in enumerate(features, 1):
    print(f"{i:2d}. {f}")

### ✍️ Your boundaries (around capabilities)

Group the feature numbers under capability boundaries — and name the **interface** each boundary exposes to the others. *(Sample grouping; revise to your own.)*

- **Agent runtime** — features 1, 5, 9 · *interface:* `run(turn) -> stream` ; reads memory, calls retrieval as a dependency.
- **Retrieval** — features 2, 7 · *interface:* `search(query) -> chunks`, `index(doc)`.
- **Billing & metering** — features 3, 8 · *interface:* `record_usage(event)`, `invoice(tenant)`.
- **Identity & access** — features 4, 6 · *interface:* `authenticate(key) -> tenant`, `check_quota(tenant)`.
- **Observability** — feature 10 · *cross-cutting:* every boundary emits traces; it does not own business logic.

> **Self-check:** would a typical change ("add a new tool", "change the invoice format") stay inside *one* boundary? If a change forces edits across three boundaries, the boundary is in the wrong place — the §27.5 source of most architectural pain.

## 🔧 Write a full ADR (ADR-014 shape)

Now *record the why* for an expensive decision — ideally the one you reached in 27-01 (*modular monolith vs microservices for the agent platform*). An ADR is short, version-controlled, and append-only: **Status / Context / Decision / Consequences** (§27.7). Fill in the template below in your own words. This lifts directly into [`templates/adr-template/`](../../templates/adr-template/).

Here is the book's **ADR-014** as the reference shape to match:

```markdown
# ADR-014: Use a modular monolith for the agent platform

## Status
Accepted — 2026-06-09

## Context
Two engineers, pre-launch, requirements still shifting. We need fast iteration
and one simple deploy, but also clean seams so we can extract services later.

## Decision
Build a modular monolith: strict module boundaries (agents, retrieval, billing)
with no cross-module imports except through defined interfaces; one deployable.

## Consequences
+ Fast to build, simple to run and debug; easy to refactor boundaries early.
- Requires discipline to keep modules separate; shared DB needs care.
- Revisit and extract services when a module needs independent scaling or a team.
```

### ✍️ Your ADR — fill in every section

```markdown
# ADR-0XX: <one-line decision, stated as a choice>

## Status
<Proposed | Accepted | Superseded by ADR-0YY> — <date>

## Context
<The forces, ranked. The top -ilities for THIS system and the real constraints
(team size, deadline, budget, scale, skills). State what makes this expensive to
reverse — why it deserves an ADR at all.>

## Decision
<The choice, concretely. What you will build, and the rule that enforces the
boundary (e.g. "no cross-module imports except through defined interfaces").>

## Consequences
+ <What gets better. Be specific.>
- <What it costs / what you give up — the "at the expense of what?" answer.>
- <The trigger that would make you revisit this decision.>

## Alternatives considered
<The 2-3 options from 27-01 you rejected, and the one-line reason each lost.
An ADR with no rejected alternatives didn't analyze a trade-off.>
```

## C4 sketch: Context and Container levels (§27.7)

C4 describes structure at four zoom levels — **Context → Container → Component → Code** — so you show the right altitude to the right audience. Sketch the **top two** levels for the capstone. Note *where human judgment concentrates* (these top levels — what systems and services exist, and why) versus where code is increasingly machine-generated (the bottom Code level) (§27.7, §27.8).

**Level 1 — Context:** who/what uses the system, and which external systems it talks to.
**Level 2 — Container:** the deployable/runnable pieces *inside* the system boundary.

### ✍️ Level 1 — Context (people + external systems)

| Element | Type | Relationship to the agent platform |
|---|---|---|
| Developer / API client | Person/System | Sends prompts, receives streamed answers |
| *<add: who else uses it?>* | Person | *<why do they touch it?>* |
| LLM provider (Anthropic) | External system | Agent platform calls it for generation |
| Vector store (Chroma/Pinecone) | External system | Retrieval reads/writes embeddings |
| Payment provider | External system | Billing charges cards |
| *<add: any other external dependency?>* | External system | *<what for?>* |

### ✍️ Level 2 — Container (runnable pieces inside the boundary)

| Container | Tech (example) | Responsibility | Talks to |
|---|---|---|---|
| API service | FastAPI + Uvicorn | HTTP entry, auth, streaming | Agent runtime, Identity |
| Agent runtime | Python module | Plan → tools → answer | LLM provider, Retrieval, Memory |
| Retrieval | Python module | Search/index documents | Vector store |
| Billing & metering | Python module | Usage events, invoices | Relational DB, Payment provider |
| *<add the container(s) you'd draw>* | *<tech>* | *<responsibility>* | *<dependencies>* |

> **Altitude check:** the Context and Container choices above are *your* judgment calls — no model can decide them for your business. The Code level (the functions inside each container) is exactly where you'd hand work to AI. Keep your design effort at the top.

## (Optional) Render the C4 sketch as a diagram

The *only* code worth running here, and it's optional: emit your Container sketch as **Mermaid** source so the artifact is shareable (GitHub renders Mermaid in Markdown natively). This needs no extra package — it just prints text. Edit the `containers`/`edges` to match the sketch you filled in above.

In [ ]:
# Build a Mermaid diagram from the sketch. No renderer needed — we print the source,
# which GitHub/Obsidian/Mermaid-live render directly. Degrades gracefully everywhere.
containers = {
    "api": "API service [FastAPI]",
    "agent": "Agent runtime",
    "retrieval": "Retrieval",
    "billing": "Billing & metering",
    "llm": "LLM provider [external]",
    "vectors": "Vector store [external]",
    "pay": "Payment provider [external]",
}
edges = [
    ("api", "agent", "runs a turn"),
    ("agent", "llm", "generation"),
    ("agent", "retrieval", "grounding"),
    ("retrieval", "vectors", "search/index"),
    ("api", "billing", "record usage"),
    ("billing", "pay", "charge"),
]

lines = ["flowchart LR"]
for key, label in containers.items():
    lines.append(f'    {key}["{label}"]')
for src, dst, label in edges:
    lines.append(f"    {src} -->|{label}| {dst}")
mermaid = "\n".join(lines)

print("```mermaid")
print(mermaid)
print("```")
print("\n# Paste the block above into a .md file or https://mermaid.live to render it.")

**What you just saw.** A C4 Container diagram you can drop into the system-design doc — no drawing tool, no binary blob, just text that diffs cleanly and renders on GitHub. This is the diagram half of [`templates/system-design-doc/`](../../templates/system-design-doc/).

## 📋 Self-audit: the §27.9 architecture review checklist

Run your ADR + C4 sketch through the book's review checklist. Mark each item and, where it fails, note the fix. This is the same checklist a reviewer would use on you.

- [ ] **Requirements:** Are the top three quality attributes named *and ranked* for this system?
- [ ] **Simplicity:** Is this the simplest architecture that meets them? What was rejected?
- [ ] **Boundaries:** Are they drawn around business capabilities, so typical changes stay local?
- [ ] **Coupling:** Can each part be understood, changed, and tested without the others?
- [ ] **Trade-offs:** For each major choice, is "at the expense of what?" answered explicitly?
- [ ] **Reversibility:** Which decisions are one-way doors? Do they have the certainty to justify it?
- [ ] **Failure:** What happens when each dependency is slow or down? Is the blast radius bounded?
- [ ] **Records:** Is each expensive decision captured in an ADR with its context and why?
- [ ] **Communication:** Could a new engineer understand the system from the diagrams + ADRs alone?

### ✍️ Where did your artifacts fail the checklist, and what's the one fix for each?

## 🎯 Senior lens

Three skills compound in value as AI writes more code, and this worksheet is reps for all three (§27.8). **Taste:** recognizing a good design from a plausible-but-wrong one — the model will happily generate both, so your C4 sketch is the standard you check against. **Judgment:** making the expensive, context-dependent trade-offs the model can't — that's the ADR's `Context` and `Alternatives` sections. **Review at scale:** when code is cheap to produce, the bottleneck becomes verifying a flood of generated code is correct and coherent — which demands the strong mental model your diagrams + ADRs encode. Invest here and AI becomes a force multiplier under your direction; neglect it and you become a bottleneck reviewing output you can't evaluate.

## Recap

- **Architecture = decisions expensive to change** (§27.1); sort by *cost to reverse* and spend attention on the one-way doors.
- Draw **boundaries around business capabilities**, not technical layers, so typical changes stay local (§27.5).
- An **ADR** records a single decision and its *why* — Status / Context / Decision / Consequences — and lists the rejected alternatives (§27.7).
- A **C4** sketch shows the system at the right altitude; the **top levels are human judgment**, the code level is increasingly machine-generated (§27.7–27.8).
- Self-audit every architecture against the **§27.9 review checklist** before you ship it.

## Exercises

1. **A second ADR.** Write an ADR for a *different* expensive decision from the sort list (e.g. "Postgres vs DynamoDB for the system of record"). Name the ranked forces first, then decide — and list at least two rejected alternatives.
2. **Tighten a boundary.** Pick one capability from the boundaries drill and write the exact interface (function signatures) it exposes. Predict which *other* boundary would have to change if you altered that interface — that's your coupling.
3. **Drop the C4 a level.** For one container, sketch the **Component** level (the modules inside it). Note which components you'd let AI generate and which you'd design by hand.
4. **Review someone else's.** Take the sample ADR above and run the §27.9 checklist on *it*. Find one item it fails and propose the fix.

In [ ]:
# Exercise scratch space — your code here (or keep it prose; this is a worksheet).


In [ ]:
# Exercise scratch space — your code here.


## Next

- **Templates this seeds:** your ADR lifts into [`templates/adr-template/`](../../templates/adr-template/), and the -ilities ranking + trade-off table + C4 sketch become [`templates/system-design-doc/`](../../templates/system-design-doc/). These two templates are the through-line of this chapter.
- **Capstone:** this worksheet produces the **founding ADR and C4 sketch** for [`capstone-project/`](../../capstone-project/) — the architectural narrative behind the build (Appendix C). No code here, but it shapes every chapter that has code.
- **Book:** you'll reuse this in §42 (system design) and §51 (senior → architect: ADR/RFC practice). Next chapter, §28 (application architecture / hexagonal) turns these boundaries into a concrete in-process structure.
- **Where this leads:** every other blueprint in the repo is *reviewed* against the judgment you practiced here — taste, trade-offs, and the §27.9 checklist.